# Ejercicio 3.1 · primer pipeline RAG sobre un PDF propio

Construimos un pipeline **RAG** (Retrieval-Augmented Generation) completo sobre `IA-Agentica-Completo-ES.pdf`: un curso de IA agéntica de 866 páginas (guía docente, apuntes y laboratorio).

La idea de RAG en una frase: en vez de preguntarle al LLM «de memoria», primero **buscamos** los fragmentos del PDF más parecidos a la pregunta y se los damos al LLM como contexto para que responda con ellos.

```
PDF → páginas → limpieza → chunks → embeddings → ChromaDB
                                                    ↓
pregunta → embedding → los 4 chunks más parecidos → prompt → LLM → respuesta
```

| Paso | Qué hacemos |
|---|---|
| 1 | Dependencias, claves y (opcional) LangSmith |
| 2-3 | Ingesta: cargar el PDF y limpiar el ruido |
| 4 | Chunking: partir el texto en trozos |
| 5 | Indexar en ChromaDB con embeddings de OpenAI |
| 6 | Cadena RAG con LCEL |
| 7-8 | Probar con 10 queries y diagnosticar las que fallan |
| Bonus 1 | Variantes de chunk_size (300 / 500 / 1000) |
| Bonus 2 | El mismo pipeline con LlamaIndex |
| Bonus 3 | Prompt que cita las fuentes (página) |

## 1. Dependencias y entorno

Las librerías se instalan con `uv` (nunca con `pip`). Ya están declaradas en `pyproject.toml`, así que normalmente no hace falta ejecutar nada. Si falta alguna:

```bash
uv add langchain langchain-openai langchain-chroma langchain-community pypdf python-dotenv tiktoken
uv add llama-index-core llama-index-embeddings-openai llama-index-llms-openai   # solo Bonus 2
```

- `pypdf`: lee el PDF y extrae el texto de cada página.
- `langchain-community`: trae `PyPDFLoader`, que usa `pypdf` y nos devuelve la página como un objeto `Document`.
- `langchain-text-splitters`: trae el splitter que parte el texto en chunks (viene con `langchain`).
- `langchain-openai`: los embeddings (`text-embedding-3-small`) y el LLM (`gpt-4o-mini`).
- `langchain-chroma`: conecta LangChain con ChromaDB.
- `tiktoken`: cuenta tokens, para estimar el coste antes de pagar.
- `python-dotenv`: lee las claves del archivo `.env`.

### Claves en `.env`

`load_dotenv()` lee `.env` y deja `OPENAI_API_KEY` disponible para las librerías. La clave **nunca** se escribe en el notebook.

In [1]:
import os                    # para leer variables de entorno
import random                # para escoger chunks al azar (paso 4)
import re                    # expresiones regulares, para limpiar texto (paso 3)

import tiktoken              # cuenta tokens (estimar coste, paso 5)
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()   # lee el archivo .env y carga las claves en el entorno

# Comprobamos que la clave existe SIN imprimirla
if os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY encontrada")
else:
    print("Falta OPENAI_API_KEY en el archivo .env")

/var/folders/n_/grsdt70j7lv8zr9c12wfbb3m0000gn/T/ipykernel_37850/2413991969.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


OPENAI_API_KEY encontrada


### (Opcional) LangSmith: ver el tracing de cada pregunta

LangSmith guarda un «trace» de cada vez que invocamos la cadena: qué chunks recuperó el retriever, el prompt final, la respuesta, los tokens y la latencia. Es justo lo que necesitamos en el paso 8.

Para activarlo solo hay que añadir estas líneas al `.env` (no se toca el código):

```
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=lsv2_...
LANGSMITH_PROJECT=rag-neoland-ejercicio-3-1
```

> ⚠️ Con el tracing activo, tus preguntas y los fragmentos del PDF se envían a un servicio externo. Con este PDF no hay problema; con documentos privados, ten cuidado.

Esta celda solo informa de si está activo. Si no lo configuras, el notebook funciona igual.

In [2]:
if os.getenv("LANGSMITH_TRACING") == "true" and os.getenv("LANGSMITH_API_KEY"):
    print("LangSmith activo. Proyecto:", os.getenv("LANGSMITH_PROJECT", "default"))
else:
    print("LangSmith NO activo (el notebook funciona igual)")

LangSmith activo. Proyecto: RAG GIST


## 2. Ingesta: cargar el PDF

`PyPDFLoader(ruta).load()` devuelve **una lista con un `Document` por página**. Cada `Document` tiene dos partes:

- `page_content`: el texto de la página.
- `metadata`: un diccionario con datos de origen. Nos importa `page` (el número de página, **empezando en 0**).

> El notebook está en `notebooks/` y el PDF en la raíz del proyecto, por eso la ruta empieza por `../`.

El PDF tiene algunos avisos `Ignoring wrong pointing object ...`: son referencias internas rotas del archivo. No afectan al texto.

In [3]:
ruta_pdf = "../IA-Agentica-Completo-ES.pdf"

cargador = PyPDFLoader(ruta_pdf)
paginas = cargador.load()

print(len(paginas), "páginas cargadas")
print(paginas[0].metadata)

Ignoring wrong pointing object 164 0 (offset 0)


Ignoring wrong pointing object 410 0 (offset 0)


Ignoring wrong pointing object 411 0 (offset 0)


Ignoring wrong pointing object 412 0 (offset 0)


Ignoring wrong pointing object 413 0 (offset 0)


Ignoring wrong pointing object 414 0 (offset 0)


... (883 outputs left out)

Ignoring wrong pointing object 2776 0 (offset 0)


Ignoring wrong pointing object 2777 0 (offset 0)


Ignoring wrong pointing object 2778 0 (offset 0)


866 páginas cargadas
{'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '../IA-Agentica-Completo-ES.pdf', 'total_pages': 866, 'page': 0, 'page_label': '1'}


Verificamos que el texto se extrae bien: imprimimos los primeros 500 caracteres de varias páginas **de distintas zonas del PDF** (portada, índice, apuntes con código, laboratorio). Si viéramos caracteres raros o palabras partidas, probaríamos `PDFPlumberLoader` o `UnstructuredPDFLoader`.

In [4]:
numeros_de_pagina = [0, 30, 32, 200, 420, 700]

for numero in numeros_de_pagina:
    print(f"===== página {numero} =====")
    print(paginas[numero].page_content[:500])
    print()

===== página 0 =====
IA Agéntica
Fundamentos, Arquitecturas y Aplicaciones
Material Docente Completo
Francisco-Javier Rodrigo-Ginés
Versión 1.0.0 — Marzo 2026
Licencia CC BY-SA 4.0

===== página 30 =====
IA Agéntica: Fundamentos, Arquitecturas y Aplicaciones Apuntes
A P U N T E S
Contenidos
Semana 1: Introducción a los agentes de IA . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4
Objetivos de aprendizaje . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4
1. Qué es un agente de IA? .

===== página 32 =====
IA Agéntica: Fundamentos, Arquitecturas y Aplicaciones Apuntes
A P U N T E S
Semana 1: Introducción a los agentes de IA
Objetivos de aprendizaje
Al finalizar esta clase, el estudiantado será capaz de:
1. Definir qué es un agente de IA y distinguir los agentes de los siste

### Qué hemos visto

El texto se extrae **bien** (tildes, ñ y código Python correctos), así que no hace falta cambiar de loader. Pero hay **ruido repetido** que va a ensuciar los chunks:

1. **Cabecera** en todas las páginas: `IA Agéntica: Fundamentos, Arquitecturas y Aplicaciones Apuntes`.
2. **Pie** en las páginas de apuntes: `Francisco-Javier Rodrigo-Ginés 4 / 35 CC BY-SA 4.0`.
3. **Índice** con filas de puntos suspensivos (`Objetivos de aprendizaje . . . . . . . . 4`).

Si no lo quitamos, cada chunk empezará con la misma frase y los embeddings se parecerán todos un poco entre sí.

## 3. Limpiar cabeceras, pies y puntos suspensivos

Una función corta que recorre el texto **línea a línea** con un `for` normal:

- Si la línea es la cabecera o el pie, la saltamos (`continue` = «pasa a la siguiente vuelta»).
- Al final, sustituimos las filas de puntos del índice con `re.sub`.

> **Nuevo:** `re.sub(patron, reemplazo, texto)` busca en el texto todo lo que encaja con una expresión regular y lo cambia. El patrón `(\. ){3,}` significa «un punto seguido de un espacio, repetido 3 o más veces».

Limpiamos `page_content` **en sitio** y dejamos la `metadata` intacta (necesitaremos `page` para citar fuentes en el Bonus 3).

In [5]:
def limpiar_texto(texto):
    lineas_limpias = []

    for linea in texto.split("\n"):
        if linea.startswith("IA Agéntica: Fundamentos"):   # cabecera repetida
            continue
        if "CC BY-SA 4.0" in linea:                         # pie de página
            continue
        lineas_limpias.append(linea)

    texto_limpio = "\n".join(lineas_limpias)
    texto_limpio = re.sub(r"(\. ){3,}", " ... ", texto_limpio)   # filas de puntos del índice
    return texto_limpio


# Guardamos cuántos caracteres había antes para ver cuánto ruido quitamos
caracteres_antes = 0
caracteres_despues = 0

for pagina in paginas:
    caracteres_antes = caracteres_antes + len(pagina.page_content)
    pagina.page_content = limpiar_texto(pagina.page_content)
    caracteres_despues = caracteres_despues + len(pagina.page_content)

print(f"Caracteres antes:   {caracteres_antes:,}")
print(f"Caracteres después: {caracteres_despues:,}")
print()
print(paginas[32].page_content[:400])

Caracteres antes:   1,880,219
Caracteres después: 1,606,666

A P U N T E S
Semana 1: Introducción a los agentes de IA
Objetivos de aprendizaje
Al finalizar esta clase, el estudiantado será capaz de:
1. Definir qué es un agente de IA y distinguir los agentes de los sistemas de software convencionales.
2. Trazar la evolución histórica desde los sistemas expertos hasta los agentes modernos basados en LLM.
3. Clasificar agentes según la taxonomía de Russell y N


## 4. Chunking con `RecursiveCharacterTextSplitter`

Un embedding representa **un solo texto** como un vector. Si le damos 3 páginas, el vector es una mezcla de todo y no sirve para buscar algo concreto. Por eso partimos el texto en **chunks** pequeños.

`RecursiveCharacterTextSplitter` intenta cortar por separadores «naturales», en este orden: salto de párrafo → salto de línea → espacio → carácter. Así evita cortar a mitad de frase siempre que puede.

| Parámetro | Valor | Qué significa |
|---|---|---|
| `chunk_size` | 500 | máximo de **caracteres** por chunk |
| `chunk_overlap` | 80 | caracteres que se repiten entre un chunk y el siguiente, para no perder una idea que caiga justo en el corte |

`split_documents(paginas)` parte cada página y **copia su `metadata`** a cada chunk (así sabemos de qué página viene).

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
chunks_todos = splitter.split_documents(paginas)

# Quitamos chunks casi vacíos (portadas, páginas con solo un título): no aportan nada
# y OpenAI da error si le mandas un texto vacío.
chunks = []
for chunk in chunks_todos:
    if len(chunk.page_content.strip()) >= 30:
        chunks.append(chunk)

print(len(chunks_todos), "chunks en total")
print(len(chunks), "chunks útiles (>= 30 caracteres)")

4040 chunks en total
4039 chunks útiles (>= 30 caracteres)


Leemos 4 chunks **al azar** como si fuéramos el LLM, sin ver el resto del documento. Preguntas a hacerse:

- ¿Se entiende solo, es **autocontenido**?
- ¿Empieza o termina a mitad de una frase o de un bloque de código?
- ¿Tiene sentido leído aislado?

`random.sample(lista, 4)` escoge 4 elementos distintos al azar. Vuelve a ejecutar la celda varias veces para ver más ejemplos.

In [7]:
for chunk in random.sample(chunks, 4):
    pagina_pdf = chunk.metadata["page"] + 1   # page empieza en 0; sumamos 1 para la página «real»
    print(f"===== página {pagina_pdf} | {len(chunk.page_content)} caracteres =====")
    print(chunk.page_content)
    print()

===== página 131 | 485 caracteres =====
}
        ],
        temperature=0.0,
    )
    text = response.choices[0].message.content
    # Parse XML-tagged sections
    analysis = re.search(r'<analysis>(.*?)</analysis>', text, re.DOTALL)
    issues = re.search(r'<issues>(.*?)</issues>', text, re.DOTALL)
    verdict = re.search(r'<verdict>(.*?)</verdict>', text, re.DOTALL)
    return {
        "analysis": analysis.group(1).strip() if analysis else "",
        "issues": json.loads(issues.group(1).strip()) if issues else [],

===== página 312 | 488 caracteres =====
return chunks[:top_k]
    def build_prompt(
        self, query: str, retrieved_chunks: list[dict]
    ) -> str:
        """Build the augmented prompt with retrieved context.
        This is the AUGMENT phase: we construct a prompt that includes
        both the user's question and the retrieved context. The prompt
        instructs the LLM to answer based on the context and to
        acknowledge when the context is insufficient

### Notas de lectura

Los chunks salen al azar, así que en cada ejecución verás otros. Esto es lo que vi en los que quedaron guardados arriba y en otras ejecuciones:

- Los chunks de **texto explicativo** son bastante autocontenidos (por ejemplo, el de la sección «3.5 Agentes con aprendizaje», pág. 44).
- Los de **código Python** se cortan a mitad de función: el chunk de la pág. 419 empieza con un `}` suelto y el de la pág. 857 termina a mitad de un bloque. 500 caracteres es poco para una función entera.
- Algunos chunks **empiezan a mitad de frase** («entrenado, tareas que requieren…», pág. 100) por el solapamiento y el corte por tamaño.
- Hay chunks con **títulos pegados al final** (`3. Evaluación basada en proceso / Más allá de las salidas finales`): el título queda separado del texto que le sigue.

Conclusión: 500 caracteres funciona para texto, pero parte código y tablas. En el Bonus 1 probamos otros tamaños.

## 5. Indexar en ChromaDB con embeddings de OpenAI

### Estimar el coste antes de pagar

Los embeddings de OpenAI se pagan por **token** (un token ≈ 4 caracteres). Con `tiktoken` contamos los tokens de todos los chunks y multiplicamos por el precio. `text-embedding-3-small` costaba **0,02 USD por millón de tokens** al escribir esto (compruébalo en la web de OpenAI, los precios cambian).

In [8]:
codificador = tiktoken.encoding_for_model("text-embedding-3-small")

tokens_totales = 0
for chunk in chunks:
    tokens_del_chunk = len(codificador.encode(chunk.page_content))
    tokens_totales = tokens_totales + tokens_del_chunk

PRECIO_POR_MILLON_DE_TOKENS = 0.02   # USD, text-embedding-3-small
coste_estimado = tokens_totales / 1_000_000 * PRECIO_POR_MILLON_DE_TOKENS

print(f"{len(chunks)} chunks, {tokens_totales:,} tokens")
print(f"Coste estimado de embeddings: {coste_estimado:.4f} USD")

4039 chunks, 442,494 tokens
Coste estimado de embeddings: 0.0088 USD


### Dos funciones: abrir la colección e indexar

Como en el ejercicio 2.2, usamos **coseno** como métrica y guardamos la base **en disco** (`./chroma_db`, la misma carpeta de antes pero con otra colección, así que no se mezcla con las noticias).

- `abrir_coleccion(nombre)`: abre (o crea) una colección de Chroma conectada al modelo de embeddings.
- `crear_indice(chunks, nombre)`: mete los chunks **en lotes de 200** (así no mandamos 6.000 textos de golpe a OpenAI).

**Para no pagar dos veces:** si la colección ya tiene todos los chunks, la reutilizamos sin llamar a OpenAI. Si se quedó a medias (por ejemplo, por cortar la celda), la borramos y empezamos de cero.

> `vectorstore._collection.count()` cuenta cuántos documentos hay en la colección. El `_` delante indica que es un atributo interno de LangChain; lo usamos porque es la forma más corta de contarlos.

In [9]:
modelo_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

TAMANO_LOTE = 200


def abrir_coleccion(nombre_coleccion):
    vectorstore = Chroma(
        collection_name=nombre_coleccion,
        embedding_function=modelo_embeddings,
        persist_directory="./chroma_db",
        collection_metadata={"hnsw:space": "cosine"},   # métrica coseno (fija para esta colección)
    )
    return vectorstore


def crear_indice(chunks, nombre_coleccion):
    vectorstore = abrir_coleccion(nombre_coleccion)
    ya_indexados = vectorstore._collection.count()

    if ya_indexados == len(chunks):
        print(f"'{nombre_coleccion}': ya estaba indexada ({ya_indexados} chunks). No pagamos embeddings otra vez.")
        return vectorstore

    if ya_indexados > 0:
        # Quedó a medias: empezamos de cero para no dejar chunks duplicados o faltantes
        vectorstore.delete_collection()
        vectorstore = abrir_coleccion(nombre_coleccion)

    for inicio in range(0, len(chunks), TAMANO_LOTE):
        lote = chunks[inicio : inicio + TAMANO_LOTE]   # trozo de la lista: de inicio a inicio+200
        vectorstore.add_documents(lote)
        if (inicio // TAMANO_LOTE) % 10 == 0:          # // = división entera; imprimimos cada 10 lotes
            print(f"   {min(inicio + TAMANO_LOTE, len(chunks))} / {len(chunks)} chunks indexados")

    print(f"'{nombre_coleccion}': {vectorstore._collection.count()} chunks indexados")
    return vectorstore

In [10]:
vectorstore = crear_indice(chunks, "ia_agentica_500")

'ia_agentica_500': ya estaba indexada (4039 chunks). No pagamos embeddings otra vez.


Comprobación rápida: pedimos los 3 chunks más parecidos a una pregunta. `similarity_search_with_score` devuelve pares `(chunk, distancia)`. Con coseno, **distancia = 1 − similitud**, así que convertimos para leerlo mejor (más cerca de 1 = más parecido).

In [11]:
pregunta_prueba = "¿Cuáles son los cuatro componentes de un agente basado en LLM?"
resultados = vectorstore.similarity_search_with_score(pregunta_prueba, k=3)

for chunk, distancia in resultados:
    similitud = 1 - distancia
    print(f"similitud {similitud:.3f} | página {chunk.metadata['page'] + 1}")
    print(chunk.page_content[:200].replace("\n", " "))
    print()

similitud 0.725 | página 31
3.2 Agentes de reflejo basados en modelo  ... 12 3.3 Agentes basados en objetivos  ... 13 3.4 Agentes basados en utilidad  ... 14 3.5 Agentes con aprendizaje  ... 15 4. El agente de IA moderno: el LLM

similitud 0.724 | página 62
4. Los agentes basados en LLM usan un modelo de lenguaje de gran escala como núcleo de razonamiento,  complementado con cuatro componentes clave: planificación (descomposición de tareas), memoria (al-

similitud 0.688 | página 323
requiere la formalización explícita de todo el conocimiento y los objetivos. Sistemas multiagente basados en LLM Los sistemas multiagente basados en LLM difieren fundamentalmente de los MÁS clásicos. 



## 6. La cadena RAG con LCEL

**LCEL** (LangChain Expression Language) encadena piezas con el operador `|`, como una tubería: la salida de cada pieza es la entrada de la siguiente.

```
pregunta ──┬─► retriever ─► formatear_chunks ─► "contexto" ─┐
           └──────────────────────────────────► "pregunta" ─┴─► prompt ─► LLM ─► texto
```

Las piezas:

- **retriever**: `vectorstore.as_retriever(search_kwargs={"k": 4})` busca los 4 chunks más parecidos a la pregunta.
- **formatear_chunks**: une el texto de esos 4 chunks en un solo string (el «contexto»).
- **`RunnablePassthrough()`**: deja pasar la pregunta tal cual, sin cambiarla.
- **prompt**: plantilla con dos huecos, `{contexto}` y `{pregunta}`. Las instrucciones importantes son dos: **responde solo con el contexto** y **si no está, dilo**. Sin eso, el LLM tiende a inventar.
- **`ChatOpenAI(model="gpt-4o-mini", temperature=0)`**: el LLM. `temperature=0` = respuestas lo más deterministas posible (mejor para comparar pruebas).
- **`StrOutputParser()`**: convierte la respuesta del LLM en un string normal.

> La llave `{"contexto": ..., "pregunta": ...}` es un diccionario: LCEL ejecuta las dos ramas con la misma pregunta y junta los resultados en un dict, que es lo que espera el prompt.

Lo metemos en una función `crear_cadena` para poder reutilizarla en los bonus con otra colección u otro prompt.

In [12]:
def formatear_chunks(chunks_recuperados):
    textos = []
    for chunk in chunks_recuperados:
        textos.append(chunk.page_content)
    return "\n\n".join(textos)   # separamos los chunks con una línea en blanco


prompt_base = ChatPromptTemplate.from_template(
    """Eres un asistente que responde preguntas sobre un curso de IA agéntica.
Responde SOLO con la información del contexto. Si el contexto no contiene la respuesta,
di exactamente: "No encuentro esa información en los documentos."

Contexto:
{contexto}

Pregunta: {pregunta}

Respuesta:"""
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def crear_cadena(vectorstore, prompt, funcion_formatear=formatear_chunks):
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
    cadena = (
        {"contexto": retriever | funcion_formatear, "pregunta": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return cadena


cadena_rag = crear_cadena(vectorstore, prompt_base)

print(cadena_rag.invoke("¿Qué es un agente de IA?"))

Un agente de IA es un sistema que percibe su entorno a través de sensores, razona sobre lo que percibe y ejecuta acciones a través de actuadores para alcanzar objetivos.


## 7. Probar con 10 queries reales

Cinco tipos de pregunta, para ver **dónde se rompe** el sistema:

| Tipo | Qué prueba |
|---|---|
| factual | respuesta clara en un sitio concreto |
| multi-fuente | hay que juntar información de varios chunks |
| jerga | términos técnicos del dominio |
| fuera del corpus | la respuesta NO está en el PDF: ¿lo reconoce o inventa? |
| typos | faltas de ortografía o formulación rara |

> Las preguntas están basadas en el índice del PDF. Cambia las que quieras por otras tuyas.

In [13]:
queries = [
    ("factual", "¿Qué dimensiones se usan para clasificar los entornos de un agente?"),
    ("factual", "¿Qué tipos de agentes distingue la taxonomía de Russell y Norvig?"),
    ("factual", "¿Cuáles son los cuatro componentes fundamentales de un agente basado en LLM?"),
    ("multi-fuente", "Compara los agentes basados en objetivos con los basados en utilidad y con los agentes con aprendizaje."),
    ("multi-fuente", "¿Qué problemas éticos plantean los agentes y cómo se relacionan con la autonomía y la responsabilidad?"),
    ("jerga", "¿Qué es MCP y para qué sirve junto con el function calling?"),
    ("jerga", "¿Qué diferencia hay entre un agente BDI y un agente de aprendizaje por refuerzo?"),
    ("fuera del corpus", "¿Cuál es la receta de la paella valenciana?"),
    ("fuera del corpus", "¿Quién ganó el Mundial de fútbol de 2022?"),
    ("typos", "que es un agnete de reflejo simpel y cuando no sirbe?"),
]

for tipo, pregunta in queries:
    respuesta = cadena_rag.invoke(pregunta)
    print(f"[{tipo}] {pregunta}")
    print(respuesta)
    print("-" * 80)

[factual] ¿Qué dimensiones se usan para clasificar los entornos de un agente?
Russell y Norvig (2021) clasifican los entornos a lo largo de varias dimensiones.
--------------------------------------------------------------------------------


[factual] ¿Qué tipos de agentes distingue la taxonomía de Russell y Norvig?
La taxonomía de Russell y Norvig distingue cinco tipos de agentes: agentes de reflejo simple, agentes basados en modelo, agentes basados en objetivos, agentes basados en utilidad y agentes con aprendizaje.
--------------------------------------------------------------------------------


[factual] ¿Cuáles son los cuatro componentes fundamentales de un agente basado en LLM?
Los cuatro componentes fundamentales de un agente basado en LLM son: planificación, memoria, uso de herramientas y acción.
--------------------------------------------------------------------------------


[multi-fuente] Compara los agentes basados en objetivos con los basados en utilidad y con los agentes con aprendizaje.
No encuentro esa información en los documentos.
--------------------------------------------------------------------------------


[multi-fuente] ¿Qué problemas éticos plantean los agentes y cómo se relacionan con la autonomía y la responsabilidad?
Los problemas éticos que plantean los agentes se relacionan con la autonomía y la responsabilidad de varias maneras. Cuando se diseña un sistema que busca aumentar la autonomía de los agentes, se enfrenta al dilema de garantizar la seguridad y la supervisión. Esto implica decisiones éticas sobre cuánta autonomía otorgar a un agente y cómo manejar las consecuencias de sus decisiones. Además, se deben considerar aspectos como la responsabilidad civil por las acciones autónomas de los agentes, la equidad en la toma de decisiones, y la privacidad de los datos que manejan. Las consideraciones éticas sobre autonomía, responsabilidad, transparencia y sesgo son centrales para el diseño responsable de agentes y se vuelven más relevantes a medida que estos asumen tareas de mayor trascendencia.
--------------------------------------------------------------------------------


[jerga] ¿Qué es MCP y para qué sirve junto con el function calling?
MCP (Multi-Client Protocol) es un protocolo que permite la reutilización de herramientas en diferentes modelos de IA, como Claude, GPT-4, Gemini y Llama. Su ventaja clave sobre el function calling directo es que un desarrollador de herramientas puede escribir un servidor MCP que funcione con múltiples modelos, evitando la necesidad de reimplementar la misma herramienta para cada proveedor. MCP estandariza las herramientas, que tienen un nombre, una descripción y un JSON Schema que define sus parámetros de entrada, permitiendo al modelo decidir cuándo llamar a cada herramienta de manera similar al function calling nativo.
--------------------------------------------------------------------------------


[jerga] ¿Qué diferencia hay entre un agente BDI y un agente de aprendizaje por refuerzo?
No encuentro esa información en los documentos.
--------------------------------------------------------------------------------


[fuera del corpus] ¿Cuál es la receta de la paella valenciana?
No encuentro esa información en los documentos.
--------------------------------------------------------------------------------


[fuera del corpus] ¿Quién ganó el Mundial de fútbol de 2022?
No encuentro esa información en los documentos.
--------------------------------------------------------------------------------


[typos] que es un agnete de reflejo simpel y cuando no sirbe?
Un agente de reflejo simple selecciona acciones basándose en reglas condición-acción aplicadas solo a la percepción actual, sin estado interno. No sirve en entornos parcialmente observables, no puede compensar fallos en los sensores, no puede razonar sobre las consecuencias de las acciones y no puede manejar situaciones no cubiertas por sus reglas.
--------------------------------------------------------------------------------


### Tabla de resultados

✅ bien · ⚠️ a medias · ❌ mal. Resultado de la ejecución guardada arriba (con `temperature=0` suele repetirse, pero no está garantizado).

| # | Tipo | ¿Resultado? | Comentario |
|---|---|---|---|
| 1 | factual (dimensiones del entorno) | ⚠️ | Solo dice «clasifican los entornos a lo largo de varias dimensiones», **sin nombrarlas**. La tabla con las dimensiones (págs. 34-35) quedó en otro chunk. |
| 2 | factual (taxonomía R&N) | ✅ | Los cinco tipos, correctos. |
| 3 | factual (4 componentes) | ✅ | Planificación, memoria, uso de herramientas y acción. |
| 4 | multi-fuente (objetivos / utilidad / aprendizaje) | ❌ | «No encuentro esa información», pero la información existe (págs. 42-44). |
| 5 | multi-fuente (ética) | ✅ | Autonomía, responsabilidad, transparencia y sesgo. |
| 6 | jerga (MCP) | ⚠️ | El desarrollo es correcto, pero **inventa la sigla**: «Multi-Client Protocol» (es *Model Context Protocol*). |
| 7 | jerga (BDI vs RL) | ❌ | «No encuentro esa información», pero el PDF trata ambos (BDI en pág. 39). |
| 8 | fuera del corpus (paella) | ✅ | Reconoce que no está. |
| 9 | fuera del corpus (Mundial 2022) | ✅ | Reconoce que no está. |
| 10 | typos | ✅ | Entiende «agnete», «simpel» y «sirbe» sin problema. |

**Balance:** 6 ✅, 2 ⚠️, 2 ❌. Las dos preguntas fuera del corpus se manejan bien gracias a la instrucción del prompt; los fallos están en las preguntas que combinan varios conceptos.

## 8. Diagnosticar las queries mal respondidas

Cuando una respuesta falla, hay **tres puntos de fallo** posibles. Se miran **en este orden**, imprimiendo los chunks que recuperó el retriever:

| # | Punto de fallo | Cómo se ve | Arreglo típico |
|---|---|---|---|
| 1 | **Recuperación**: el chunk con la respuesta NO está entre los 4 | la respuesta correcta no aparece en ningún chunk | subir `k`, cambiar chunk_size, mejorar el retriever (híbrido, reranking) |
| 2 | **Chunking**: la respuesta está partida entre dos chunks o sin contexto | aparece la mitad de la idea | cambiar `chunk_size` / `overlap` / splitter |
| 3 | **Generación**: los chunks correctos SÍ están pero el LLM responde mal | la respuesta está en los chunks y aun así falla | mejorar el prompt, usar otro modelo |

La función imprime los 4 chunks con su **similitud** y su página. Una pista para las preguntas fuera del corpus: si la similitud del mejor chunk es baja, el sistema debería poder decir «no sé», pero un RAG básico **siempre** recupera 4 chunks, sean relevantes o no.

Escribe en `queries_a_diagnosticar` las que te hayan salido mal en el paso 7.

In [14]:
def ver_chunks_recuperados(pregunta, k=4):
    resultados = vectorstore.similarity_search_with_score(pregunta, k=k)
    print("PREGUNTA:", pregunta)
    for posicion, (chunk, distancia) in enumerate(resultados, start=1):
        similitud = 1 - distancia
        print(f"  #{posicion} similitud {similitud:.3f} | página {chunk.metadata['page'] + 1}")
        print("     ", chunk.page_content[:300].replace("\n", " "))
    print()


# Cambia esta lista por las queries que fallaron en tu ejecución
queries_a_diagnosticar = [
    "Compara los agentes basados en objetivos con los basados en utilidad y con los agentes con aprendizaje.",
    "¿Qué diferencia hay entre un agente BDI y un agente de aprendizaje por refuerzo?",
    "¿Cuál es la receta de la paella valenciana?",
]

for pregunta in queries_a_diagnosticar:
    ver_chunks_recuperados(pregunta)

PREGUNTA: Compara los agentes basados en objetivos con los basados en utilidad y con los agentes con aprendizaje.
  #1 similitud 0.720 | página 62
      importante. 3. Correspondencia con Russell y Norvig: Dónde encajan los agentes modernos basados en LLM como  Claude Code en la taxonomía de Russell y Norvig? Son agentes basados en objetivos? Basados en utilidad?  Agentes con aprendizaje? Podrían ser todos estos a la vez? Punto de partida: Considera
  #2 similitud 0.699 | página 42
      de conversación sirve como modelo interno del agente sobre la tarea en curso. Ejemplos reales: Sistemas de hogar inteligente, control de crucero adaptativo, sistemas antibloqueo de frenos,  sistemas de gestión de inventario. 3.3 Agentes basados en objetivos Los agentes basados en objetivos van más a
  #3 similitud 0.680 | página 44
      seguridad, comodidad), optimización de carteras, sistemas de precios dinámicos. 3.5 Agentes con aprendizaje Un agente con aprendizáje tiene cuatro componentes conceptu

PREGUNTA: ¿Qué diferencia hay entre un agente BDI y un agente de aprendizaje por refuerzo?
  #1 similitud 0.612 | página 39
      es reactiva (condicionada al contexto inmediato), pero las estrategias de prompting como Chain-of-Thought  añaden una capa deliberativa encima. 2.4 Agentes de aprendizaje por refuerzo (década de 2010) La combinación de aprendizaje profundo y aprendizaje por refuerzo produjo agentes capaces de aprend
  #2 similitud 0.600 | página 40
      Idea clave: Los agentes de RL aprenden de la experiencia; los agentes basados en LLM aprenden de  texto sobre la experiencia. Un agente de RL que juega al ajedrez aprende jugando millones de partidas.  Un LLM "aprende" sobre ajedrez leyendo libros, artículos y transcripciones de partidas escritos po
  #3 similitud 0.597 | página 44
      seguridad, comodidad), optimización de carteras, sistemas de precios dinámicos. 3.5 Agentes con aprendizaje Un agente con aprendizáje tiene cuatro componentes conceptuales: 1. Elemento de ap

PREGUNTA: ¿Cuál es la receta de la paella valenciana?
  #1 similitud 0.295 | página 197
      lo largo de muchos pasos. Consideremos una analogía. El bucle simple del agente es como cocinar mirando los ingredientes que tienes  delante y decidiendo qué hacer a continuación. Funciona para huevos revueltos. Pero para una cena de cinco  platos para 30 personas, se necesita una receta (plan), un 
  #2 similitud 0.230 | página 390
      Pruébalo tú mismo: Piensa en preparar una cena completa con varios platos. ¿Qué tareas pueden  hacerse en paralelo (por ejemplo, hervir la pasta mientras se asan las verduras)? ¿Qué tareas tienen  dependencias (por ejemplo, no se puede emplatar antes de cocinar)? Dibuja un grafo de tareas para  una 
  #3 similitud 0.228 | página 834
      P A R T  V I Recursos Lista de lecturas, guía de instalación y glosario
  #4 similitud 0.227 | página 611
      L A B O R AT O R I O Construir un Agente ReAct desde  Cero Laboratorio 2 — Semanas 4-5 Francisco-Javier Rodrigo-

### Diagnóstico

Para cada query mal respondida, comprobé si la respuesta existe en el índice y en qué posición salió el chunk:

| Query | Punto de fallo | Evidencia | Posible arreglo |
|---|---|---|---|
| **1.** Dimensiones del entorno | **2 · Chunking** (y algo de 1) | La frase «clasifican los entornos a lo largo de varias dimensiones» se recupera, pero la **tabla** con las dimensiones (observabilidad, determinismo…) queda en otro chunk. En una búsqueda directa, ese chunk ni siquiera aparece entre los 6 primeros. | Chunks mayores o que respeten tablas; subir `k`. |
| **4.** Objetivos vs utilidad vs aprendizaje | **1 · Recuperación** | El chunk de «3.4 Agentes basados en utilidad» (pág. 43) **existe** y sale primero si se busca con términos más específicos, pero con la pregunta original no entra en el top-4. Entran en su lugar un chunk de preguntas de discusión (pág. 62) y los objetivos de aprendizaje (pág. 33). El LLM, al ver un contexto incompleto, responde «no encuentro». | Subir `k` (por ejemplo 8) y usar **reranking**; **descomponer** la pregunta en tres. |
| **6.** MCP | **3 · Generación** | Los chunks recuperados (págs. 181-190) son buenos, pero **ninguno muestra la expansión «Model Context Protocol» escrita tal cual** (comprobado con una búsqueda de texto). El LLM la inventó aunque el prompt dice «solo con el contexto». | Prompt más estricto («no expandas siglas que no aparezcan en el contexto»); revisar que la definición esté en algún chunk (la expansión aparece, por ejemplo, en la pág. 35). |
| **7.** BDI vs refuerzo | **1 · Recuperación** | El texto de BDI **sí está indexado** (pág. 39: «Creencias, Deseos, Intenciones»), pero la pregunta mezcla dos conceptos y el embedding se inclina hacia «aprendizaje por refuerzo» (págs. 39, 40, 44 y 10). Ningún chunk del top-4 explica BDI. | **Descomponer** en dos búsquedas, **búsqueda híbrida** (BM25 acierta con la sigla «BDI») o subir `k`. |

**Patrón común:** 3 de los 4 fallos están en la recuperación o el chunking, no en el LLM. En las preguntas que combinan varios conceptos, `k=4` se queda corto: es el caso típico que resuelven las próximas lecciones (hybrid, reranking, CRAG).

**Pista sobre «no sé»:** en la pregunta de la paella el mejor chunk tiene similitud **≈0,30**, mientras que las preguntas con respuesta real superan **0,55**. Un umbral de similitud (por ejemplo 0,45) permitiría decir «no sé» sin llamar al LLM. Eso ya sería un paso hacia CRAG.

> No te frustres con las que fallen: es lo normal del **naive RAG**.

## Bonus 1 · variantes de chunking (300, 500 y 1000)

Repetimos el experimento con tres tamaños. Mantenemos el solapamiento en torno al **16 %** del tamaño para que la comparación sea justa:

| chunk_size | chunk_overlap | colección |
|---|---|---|
| 300 | 50 | `ia_agentica_300` |
| 500 | 80 | `ia_agentica_500` (ya creada) |
| 1000 | 160 | `ia_agentica_1000` |

Para cada tamaño: partimos, filtramos chunks vacíos, indexamos (cuesta céntimos) y hacemos 3 preguntas. Comparamos **cuántos chunks salen**, **la similitud del mejor chunk** y la respuesta.

> ⚠️ Cada tamaño se indexa en su propia colección: no mezclamos tamaños dentro de una misma colección.

In [15]:
variantes = [(300, 50), (500, 80), (1000, 160)]

preguntas_bonus1 = [
    "¿Cuáles son los cuatro componentes fundamentales de un agente basado en LLM?",
    "Compara los agentes basados en objetivos con los basados en utilidad y con los agentes con aprendizaje.",
    "que es un agnete de reflejo simpel y cuando no sirbe?",
]

# Aquí guardamos el vectorstore de cada tamaño para usarlo después
vectorstores_por_tamano = {}

for tamano, solapamiento in variantes:
    splitter_variante = RecursiveCharacterTextSplitter(chunk_size=tamano, chunk_overlap=solapamiento)
    chunks_variante_todos = splitter_variante.split_documents(paginas)

    chunks_variante = []
    for chunk in chunks_variante_todos:
        if len(chunk.page_content.strip()) >= 30:
            chunks_variante.append(chunk)

    print(f"chunk_size={tamano}: {len(chunks_variante)} chunks")
    vectorstores_por_tamano[tamano] = crear_indice(chunks_variante, f"ia_agentica_{tamano}")

chunk_size=300: 6786 chunks
'ia_agentica_300': ya estaba indexada (6786 chunks). No pagamos embeddings otra vez.


chunk_size=500: 4039 chunks
'ia_agentica_500': ya estaba indexada (4039 chunks). No pagamos embeddings otra vez.
chunk_size=1000: 2196 chunks
'ia_agentica_1000': ya estaba indexada (2196 chunks). No pagamos embeddings otra vez.


In [16]:
for pregunta in preguntas_bonus1:
    print("=" * 90)
    print("PREGUNTA:", pregunta)

    for tamano, solapamiento in variantes:
        vectorstore_variante = vectorstores_por_tamano[tamano]

        # Similitud del mejor chunk
        mejor_chunk, distancia = vectorstore_variante.similarity_search_with_score(pregunta, k=1)[0]
        similitud = 1 - distancia

        # Respuesta completa con esa colección
        cadena_variante = crear_cadena(vectorstore_variante, prompt_base)
        respuesta = cadena_variante.invoke(pregunta)

        print(f"\n--- chunk_size={tamano} | mejor similitud {similitud:.3f} | página {mejor_chunk.metadata['page'] + 1}")
        print(respuesta)

PREGUNTA: ¿Cuáles son los cuatro componentes fundamentales de un agente basado en LLM?



--- chunk_size=300 | mejor similitud 0.788 | página 46
Los cuatro componentes fundamentales de un agente basado en LLM son: planificación, memoria, uso de herramientas y acción.



--- chunk_size=500 | mejor similitud 0.748 | página 31
Los cuatro componentes fundamentales de un agente basado en LLM son: planificación, memoria, uso de herramientas y acción.



--- chunk_size=1000 | mejor similitud 0.702 | página 46
No encuentro esa información en los documentos.
PREGUNTA: Compara los agentes basados en objetivos con los basados en utilidad y con los agentes con aprendizaje.



--- chunk_size=300 | mejor similitud 0.737 | página 33
Los agentes basados en objetivos van más allá de reaccionar al estado actual, ya que consideran el futuro y qué acciones conducirían a alcanzar sus objetivos. Por otro lado, los agentes basados en utilidad evalúan diferentes acciones en función de una función de utilidad que mide la satisfacción de sus objetivos. Los agentes con aprendizaje, en cambio, son capaces de mejorar su rendimiento a lo largo del tiempo a través de la experiencia y la adaptación a nuevas situaciones. En resumen, los agentes basados en objetivos se centran en alcanzar metas específicas, los basados en utilidad buscan maximizar la satisfacción de esas metas, y los agentes con aprendizaje se enfocan en mejorar su capacidad para alcanzar objetivos a través de la experiencia.



--- chunk_size=500 | mejor similitud 0.720 | página 62
No encuentro esa información en los documentos.



--- chunk_size=1000 | mejor similitud 0.730 | página 43
Los agentes basados en objetivos actúan para transformar el mundo en lo que quieren que sea, mientras que los agentes basados en utilidad asignan un número real a los estados que representa cuán satisfecho está el agente en ese estado, permitiendo evaluar múltiples formas de alcanzar un objetivo y seleccionar las mejores. Por otro lado, los agentes con aprendizaje tienen cuatro componentes: un elemento de aprendizaje que mejora al agente basándose en la experiencia, un elemento de rendimiento que selecciona acciones, un crítico que proporciona retroalimentación y un generador de problemas que sugiere acciones exploratorias para descubrir nuevo conocimiento. Esto último introduce el dilema de exploración vs. explotación, lo que permite a los agentes con aprendizaje adaptarse y mejorar con el tiempo.
PREGUNTA: que es un agnete de reflejo simpel y cuando no sirbe?



--- chunk_size=300 | mejor similitud 0.625 | página 41
Un agente de reflejo simple es el tipo de agente más sencillo que selecciona acciones basándose únicamente en la percepción actual, ignorando todo el historial. No tiene memoria, no posee un modelo del mundo y no realiza planificación; solo sigue reglas de condición-acción. Este tipo de agente no sirve en situaciones donde se requiere un historial de acciones o un modelo del entorno para tomar decisiones más informadas.



--- chunk_size=500 | mejor similitud 0.599 | página 794
Un agente de reflejo simple selecciona acciones basándose en reglas condición-acción aplicadas solo a la percepción actual, sin estado interno. No sirve en entornos parcialmente observables, no puede compensar fallos en los sensores, no puede razonar sobre las consecuencias de las acciones y no puede manejar situaciones no cubiertas por sus reglas.



--- chunk_size=1000 | mejor similitud 0.492 | página 34
Un agente de reflejo simple selecciona acciones basándose en reglas condición-acción aplicadas solo a la percepción actual, sin estado interno. No sirve en situaciones donde se requiere recordar información pasada o manejar observabilidad parcial, ya que no tiene en cuenta el historial completo de percepciones.


### Notas de calibración

Número de chunks: **300 → 6.786 · 500 → 4.039 · 1000 → 2.196**.

Resultado de las 3 preguntas (✅ completa · ❌ «no encuentro»), con la similitud del mejor chunk:

| Pregunta | 300 | 500 | 1000 |
|---|---|---|---|
| 4 componentes de un agente LLM | ✅ (0,788) | ✅ (0,748) | ❌ (0,702) |
| Objetivos vs utilidad vs aprendizaje | ✅ (0,737) | ❌ (0,720) | ✅ (0,730) |
| Typos: agente de reflejo simple | ✅ (0,625) | ✅ (0,599) | ✅ (0,492) |

| chunk_size | ¿Respuestas completas? | ¿Mucho ruido en el contexto? | Veredicto |
|---|---|---|---|
| 300 | 3 de 3. La comparación sale resumida. | Poco; pero el chunk top-1 de la comparación es de la pág. 33 (objetivos de aprendizaje), no del apartado de tipos. | Mejor similitud en 2 de 3; el doble de chunks (más coste de indexado). |
| 500 | 2 de 3 | Medio | Falla justo en la comparación. |
| 1000 | 2 de 3, pero la respuesta de la comparación es **la más completa** (incluye los 4 componentes del agente con aprendizaje). | Más: la similitud baja (0,49 en la query con typos) porque el embedding mezcla temas. | Falla en la pregunta más simple. |

**Lectura honesta:** con solo 3 preguntas **no hay un ganador claro**. Cada tamaño falla en una pregunta distinta, lo que apunta a que el fallo está más en la **recuperación** (variabilidad en qué chunks entran en el top-4) que en el tamaño. Para elegir de verdad habría que medir con 20-30 preguntas y mirar si el chunk correcto aparece en el top-k, no solo la respuesta final.

Pautas que sí se confirman:

- **Chunks pequeños** (300): similitudes más altas, más precisos, pero hay que indexar más chunks.
- **Chunks grandes** (1000): contienen la idea entera (mejor para comparar conceptos), pero la similitud baja y entra más texto irrelevante.
- Este PDF mezcla texto y mucho código Python. Un único tamaño es un compromiso; cada tipo de contenido pediría el suyo.

**Decisión provisional:** nos quedamos con **500** como base de las próximas lecciones, y atacamos el problema de fondo con mejor recuperación (hybrid, reranking), no ajustando el tamaño.

## Bonus 2 · el mismo pipeline con LlamaIndex

LlamaIndex hace lo mismo que hemos construido, pero con **menos piezas visibles**: en vez de encadenar retriever + prompt + LLM a mano, `index.as_query_engine()` ya lo trae todo junto.

| Concepto | LangChain (lo de arriba) | LlamaIndex |
|---|---|---|
| Página | `Document` | `Document` |
| Partir | `RecursiveCharacterTextSplitter` | `SentenceSplitter` |
| Base vectorial | `Chroma` | `VectorStoreIndex` (en memoria por defecto) |
| Pipeline entero | cadena LCEL con `\|` | `query_engine` |

⚠️ **Cuidado con la unidad:** el `SentenceSplitter` de LlamaIndex mide `chunk_size` en **tokens**, no en caracteres. 500 caracteres ≈ 125 tokens, así que usamos `chunk_size=128` y `chunk_overlap=20` para que sea comparable con el paso 4.

Reutilizamos las mismas `paginas` (ya limpias). Convertimos cada una a un `Document` de LlamaIndex con un `for`, guardando el número de página en el metadato.

In [17]:
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# Configuración global de LlamaIndex: qué modelos usar
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)

# Pasamos las páginas de LangChain a documentos de LlamaIndex
documentos_llama = []
for pagina in paginas:
    numero_pagina = pagina.metadata["page"] + 1
    documento = Document(text=pagina.page_content, metadata={"pagina": numero_pagina})
    documentos_llama.append(documento)

splitter_llama = SentenceSplitter(chunk_size=128, chunk_overlap=20)

# Construye el índice: parte, calcula embeddings y los guarda en memoria (cuesta céntimos)
indice_llama = VectorStoreIndex.from_documents(
    documentos_llama,
    transformations=[splitter_llama],
    show_progress=True,
)
print("Índice LlamaIndex listo")

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/471 [00:00<?, ?it/s]

Índice LlamaIndex listo


In [18]:
motor_consultas = indice_llama.as_query_engine(similarity_top_k=4)

preguntas_bonus2 = [
    "¿Cuáles son los cuatro componentes fundamentales de un agente basado en LLM?",
    "¿Cuál es la receta de la paella valenciana?",
    "que es un agnete de reflejo simpel y cuando no sirbe?",
]

for pregunta in preguntas_bonus2:
    respuesta_llama = motor_consultas.query(pregunta)
    print("PREGUNTA:", pregunta)
    print("LLAMAINDEX:", respuesta_llama)

    # source_nodes = los chunks que usó para responder
    paginas_usadas = []
    for nodo in respuesta_llama.source_nodes:
        paginas_usadas.append(nodo.metadata["pagina"])
    print("Páginas usadas:", paginas_usadas)

    print("LANGCHAIN: ", cadena_rag.invoke(pregunta))
    print("-" * 80)

PREGUNTA: ¿Cuáles son los cuatro componentes fundamentales de un agente basado en LLM?
LLAMAINDEX: Los cuatro componentes fundamentales de un agente basado en LLM son: planificación, memoria, uso de herramientas y acción.
Páginas usadas: [46, 46, 323, 31]


LANGCHAIN:  Los cuatro componentes fundamentales de un agente basado en LLM son: planificación, memoria, uso de herramientas y acción.
--------------------------------------------------------------------------------


PREGUNTA: ¿Cuál es la receta de la paella valenciana?
LLAMAINDEX: No se proporciona información sobre la receta de la paella valenciana en el contenido disponible.
Páginas usadas: [155, 197, 390, 155]


LANGCHAIN:  No encuentro esa información en los documentos.
--------------------------------------------------------------------------------


PREGUNTA: que es un agnete de reflejo simpel y cuando no sirbe?
LLAMAINDEX: Un agente de reflejo simple es un tipo de agente que selecciona acciones basándose únicamente en reglas de condición-acción aplicadas a la percepción actual, sin mantener un estado interno. Esto significa que su funcionamiento se limita a responder a estímulos inmediatos sin recordar información pasada. 

No sirve en situaciones que no están cubiertas por sus reglas, lo que puede llevar a fallos en contextos más complejos donde se requiere un entendimiento más profundo o una memoria de eventos anteriores. Ejemplos de este tipo de agentes incluyen termostatos y filtros de spam simples.
Páginas usadas: [34, 794, 788, 41]


LANGCHAIN:  Un agente de reflejo simple selecciona acciones basándose en reglas condición-acción aplicadas solo a la percepción actual, sin estado interno. No sirve en entornos parcialmente observables, no puede compensar fallos en los sensores, no puede razonar sobre las consecuencias de las acciones y no puede manejar situaciones no cubiertas por sus reglas.
--------------------------------------------------------------------------------


### Comparación de experiencia

| Aspecto | LangChain | LlamaIndex |
|---|---|---|
| Líneas de código para el pipeline | ~25 (formateador, prompt, LLM, `crear_cadena`) | ~2 (`as_query_engine` + `query`) tras configurar `Settings` |
| Control sobre el prompt | Total y visible: lo escribimos nosotros (y así forzamos «solo con el contexto»). | Trae un prompt por defecto que no vemos; se puede cambiar, pero está más escondido. |
| Facilidad para ver los chunks usados | Hay que pedirlos aparte (`similarity_search_with_score`). | **Más fácil**: `respuesta.source_nodes` viene en la propia respuesta. |
| Persistencia | Chroma en disco, reutilizable. | Índice en memoria: se pierde y **hay que pagar los embeddings otra vez** al reejecutar. |
| Calidad de las respuestas | Más literal y estricta. | Similar, algo más «charlatana»: añade ejemplos (termostatos, filtros de spam) y en la paella explica por qué no hay información en vez de la frase fija. |

**Resultados en las 3 preguntas:** los dos aciertan los cuatro componentes y los dos reconocen que la paella no está. En la pregunta con typos, LangChain da las 4 limitaciones del agente de reflejo simple; LlamaIndex da solo una parte, pero ambos recuperan las mismas páginas clave (34, 41, 794).

**Conclusión:** LlamaIndex es más rápido para tener algo funcionando; LangChain deja ver y cambiar cada pieza. Como en las próximas lecciones sustituiremos el retriever y el generador por separado, **seguimos con LangChain** como base.

## Bonus 3 · prompt que cita las fuentes

Para que el LLM diga **de dónde** sale cada dato, tiene que **ver** de dónde sale. Hacemos dos cambios:

1. **Contexto formateado con metadatos:** cada chunk va precedido de `[Fuente: página N]`. Es una nueva función formateadora, `formatear_chunks_con_fuentes`.
2. **Prompt que lo exige:** le pedimos citar `(pág. N)` tras cada afirmación y, si el propio texto indica el apartado (por ejemplo «5.2 Memoria»), añadirlo.

Reutilizamos `crear_cadena`, que ya acepta un formateador y un prompt distintos. Así no reescribimos el pipeline.

In [19]:
def formatear_chunks_con_fuentes(chunks_recuperados):
    textos = []
    for chunk in chunks_recuperados:
        numero_pagina = chunk.metadata["page"] + 1
        texto_con_fuente = f"[Fuente: página {numero_pagina}]\n{chunk.page_content}"
        textos.append(texto_con_fuente)
    return "\n\n".join(textos)


prompt_con_fuentes = ChatPromptTemplate.from_template(
    """Eres un asistente que responde preguntas sobre un curso de IA agéntica.
Responde SOLO con la información del contexto. Si el contexto no contiene la respuesta,
di exactamente: "No encuentro esa información en los documentos."

Reglas para citar:
- Cada fragmento del contexto empieza con [Fuente: página N].
- Después de cada afirmación, cita la fuente así: (pág. N).
- Si el propio texto del fragmento muestra el apartado (por ejemplo "5.2 Memoria"), añádelo: (pág. N, apartado 5.2).
- No cites páginas que no aparezcan en el contexto.

Contexto:
{contexto}

Pregunta: {pregunta}

Respuesta con citas:"""
)

cadena_con_fuentes = crear_cadena(vectorstore, prompt_con_fuentes, formatear_chunks_con_fuentes)

for pregunta in [
    "¿Cuáles son los cuatro componentes fundamentales de un agente basado en LLM?",
    "¿Qué tipos de agentes distingue la taxonomía de Russell y Norvig?",
    "¿Cuál es la receta de la paella valenciana?",
]:
    print("PREGUNTA:", pregunta)
    print(cadena_con_fuentes.invoke(pregunta))
    print("-" * 80)

PREGUNTA: ¿Cuáles son los cuatro componentes fundamentales de un agente basado en LLM?


Los cuatro componentes fundamentales de un agente basado en LLM son: planificación, memoria, uso de herramientas y acción (pág. 62).
--------------------------------------------------------------------------------
PREGUNTA: ¿Qué tipos de agentes distingue la taxonomía de Russell y Norvig?


La taxonomía de Russell y Norvig distingue cinco tipos de agentes, cada uno más capaz que el anterior: 

1. Agentes de reflejo simple
2. Agentes basados en modelo
3. Agentes basados en objetivos
4. Agentes basados en utilidad
5. Agentes con aprendizaje

Esta clasificación proporciona un marco útil para comprender las capacidades de los agentes (pág. 41, apartado 3).
--------------------------------------------------------------------------------
PREGUNTA: ¿Cuál es la receta de la paella valenciana?


No encuentro esa información en los documentos.
--------------------------------------------------------------------------------


### Siguiente paso

Cuando este pipeline funcione bien, guárdalo **limpio en un módulo Python reutilizable** (por ejemplo `rag_pdf.py` en la raíz) con estas funciones: `limpiar_texto`, `crear_indice`, `crear_cadena`. En las próximas lecciones (hybrid, reranking, CRAG) sustituiremos solo el retriever o el generador, sin reescribir todo.